In [ ]:
import dotenv
import kagglehub
import pandas as pd
import plotly.graph_objects as go
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor

In [ ]:
# working_dir = "/kaggle/input/rohlik-orders-forecasting-challenge"
working_dir = "data/"

try:
    train = pd.read_csv(f"{working_dir}train.csv")
    train_calendar = pd.read_csv(f"{working_dir}train_calendar.csv")
    test = pd.read_csv(f"{working_dir}test.csv")
    test_calendar = pd.read_csv(f"{working_dir}test_calendar.csv")
except FileNotFoundError:
    dotenv.load_dotenv(dotenv.find_dotenv())
    kagglehub.login()
    kagglehub.competition_download("rohlik-orders-forecasting-challenge", output_dir = "./data")
    train = pd.read_csv(f"{working_dir}train.csv")
    train_calendar = pd.read_csv(f"{working_dir}train_calendar.csv")
    test = pd.read_csv(f"{working_dir}test.csv")
    test_calendar = pd.read_csv(f"{working_dir}test_calendar.csv")

train["date"] = pd.to_datetime(train["date"])
train = train.sort_values(by=["warehouse", "date"])
train_calendar["date"] = pd.to_datetime(train_calendar["date"])
train_calendar = train_calendar.sort_values(by=["warehouse", "date"])
test = test.sort_values(by=["warehouse", "date"])
test_calendar = test_calendar.sort_values(by=["warehouse", "date"])

#### Calendar

In [ ]:
not_provided_test_cols = ["shutdown", "mini_shutdown", "blackout", "mov_change",
                          "frankfurt_shutdown", "precipitation", "snow", "user_activity_1",
                          "user_activity_2"]

In [ ]:
train_provided_cols = train.loc[:, ~train.columns.isin(not_provided_test_cols)]
train_calendar_provided_cols = train_calendar.loc[:, ~train_calendar.columns.isin(not_provided_test_cols)]

In [ ]:
def preprocess_calendar(data: pd.DataFrame, data_calendar: pd.DataFrame) -> pd.DataFrame:
    preprocessed_calendar = pd.DataFrame()
    for warehouse in data["warehouse"].unique():
        data_calendar_w = data_calendar[data_calendar["warehouse"] == warehouse]
        data_w = data[data["warehouse"] == warehouse]
        preprocessed_calendar_w = data_calendar_w[~data_calendar_w["date"].isin(data_w["date"])]
        preprocessed_calendar = preprocessed_calendar_w if preprocessed_calendar.empty\
            else pd.concat([preprocessed_calendar, preprocessed_calendar_w])
    preprocessed_calendar["is_calendar"] = 1
    return preprocessed_calendar

train_additional_calendar = preprocess_calendar(train, train_calendar)
test_additional_calendar = preprocess_calendar(test, test_calendar)

In [ ]:
train_additional_calendar.head()

Categorical

In [ ]:
train["warehouse"].value_counts()

Numerical

In [ ]:
fig = go.Figure()
for warehouse in train['warehouse'].unique():

    train_aux = train[train['warehouse'] == warehouse]
    fig.add_trace(go.Scatter(x=train_aux['date'], y=train_aux['orders'], mode='lines+markers',
                             name=f'Orders {warehouse}'))
fig.update_layout(title=f'Total orders')
    # fig.show()
    # stock breakpoints in december (24 and 31) and 1st of january

Model

In [ ]:
train_ag = TimeSeriesDataFrame(train[["warehouse", "orders","date"]], id_column="warehouse", timestamp_column="date")
predictor = TimeSeriesPredictor(target="orders", freq="D", prediction_length=30, eval_metric="mape")

In [ ]:
predictor.fit(train_ag, presets="high_quality")

In [ ]:
predictor.leaderboard()

In [ ]:
prueba_test = train[train["date"] <= max(train["date"]) - pd.DateOffset(days=30)]
prueba_test_ag = TimeSeriesDataFrame(prueba_test, id_column="warehouse", timestamp_column="date")
predicciones = predictor.predict(prueba_test_ag)
# prueba_test_ag.head()

In [ ]:
predicciones_plot[["item_id", "timestamp", "mean"]].rename(columns={"item_id": "warehouse",
                                                                    "timestamp": "date",
                                                                    "mean": "forecast"}).head(10)

In [ ]:
predicciones_plot = predicciones.reset_index()
for warehouse in train["warehouse"].unique():
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=train[train["warehouse"] == warehouse]["date"],
                             y=train[train["warehouse"] == warehouse]["orders"],
                             mode="lines", name=warehouse))
    fig.add_trace(go.Scatter(x=predicciones_plot[predicciones_plot["item_id"] == warehouse]["timestamp"],
                             y=predicciones_plot[predicciones_plot["item_id"] == warehouse]["mean"],
                             mode="lines", name=f"{warehouse}_pred"))
    fig.update_layout(title=f"Total sales by day of the week for warehouse {warehouse}")
    fig.show()
    print(warehouse)

In [ ]:
predictions = predictor.predict(train_ag)